In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install -U transformers accelerate peft datasets bitsandbytes sentencepiece


In [ ]:
from datasets import load_dataset
dataset = load_dataset(
    "csv",
    data_files={
        "train": "/content/drive/MyDrive/clinvar_train_gene_only_30k.csv",
        "validation": "/content/drive/MyDrive/clinvar_train_gene_only_30k.csv"
    }
)
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 30000
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 30000
    })
})


In [ ]:
from huggingface_hub import login
login()

In [ ]:
from transformers import AutoTokenizer
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(tokenized)


Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 30000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 30000
    })
})


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_name = "meta-llama/Llama-3.2-3B-Instruct"

# ✅ Correct 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)
# ✅ Correct model loading (NO load_in_4bit here)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

# ✅ LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

trainable params: 2,293,760 || all params: 3,215,043,584 || trainable%: 0.0713


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/clinvar_llama3b_30k",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=2,
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"]
)


In [ ]:
trainer.train()

Step,Training Loss
100,0.872715
200,0.030932
300,0.028371
400,0.027016
500,0.024931


KeyboardInterrupt: 

In [ ]:
import os

ckpt_path = "/content/drive/MyDrive/clinvar_llama3b_30k/checkpoint-500"
print(os.listdir(ckpt_path))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'scaler.pt', 'rng_state.pth', 'trainer_state.json']


In [ ]:
FINAL_SAVE_PATH = "/content/drive/MyDrive/clinvar_llama3b_30k_final"

trainer.model.save_pretrained(FINAL_SAVE_PATH)
tokenizer.save_pretrained(FINAL_SAVE_PATH)

print("✅ Clean final adapter saved at:", FINAL_SAVE_PATH)


✅ Clean final adapter saved at: /content/drive/MyDrive/clinvar_llama3b_30k_final


In [ ]:
!pip -q install gradio transformers accelerate bitsandbytes peft sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 19.7 MB/s eta 0:00:00


In [ ]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel


In [ ]:
BASE_MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/clinvar_llama3b_30k_final"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)
base_model.eval()


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

In [ ]:
ft_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

ft_model = PeftModel.from_pretrained(ft_base_model, ADAPTER_PATH)
ft_model.eval()


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linea

In [ ]:
# @title
def generate(model, question, explain=False):
    if explain:
        instruction = """You are a clinical genomics assistant.
Explain the role of the gene(s) in the disease using clear clinical language."""
    else:
        instruction = """You are a clinical genomics assistant.
Do not expand gene names.
Do not include explanations.
Output only gene symbols."""

    prompt = f"""### Instruction:
{instruction}

### Input:
{question}

### Output:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


In [ ]:
import re

def extract_genes_only(text):
    return re.findall(r"\b[A-Z0-9]{2,8}\b", text)


In [ ]:
def explain_with_base_model(genes, disease):
    gene_str = ", ".join(genes)

    prompt = f"""### Instruction:
You are a clinical genomics assistant.
Explain the role of the gene(s) {gene_str} in {disease} using clear clinical language.
Do not introduce any new genes.

### Output:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


In [ ]:
question = "Which genes are associated with Cowden syndrome?"

genes = generate(ft_model, question, explain=False)
print("FT genes:", genes)

explanation = explain_with_base_model(genes, "Cowden syndrome")
print("Explanation:", explanation)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


FT genes: PTEN
Explanation: Cowden syndrome is an autosomal dominant genetic disorder characterized by multiple hamartomatous lesions and an increased risk of certain cancers. The syndrome is caused by mutations in the PTEN gene (P), which encodes for a tumor suppressor protein that regulates cell growth and division. Mutations in this gene lead to uncontrolled cell proliferation, resulting in the formation of benign tumors called hamartomas. These lesions can occur in various organs, including the skin, gastrointestinal tract, and breast tissue. Individuals with Cowden syndrome also have an elevated risk of developing certain types of cancer, such as thyroid, endometrial, and breast cancer. The presence of these genetic mutations increases the likelihood of developing these conditions, making early detection and monitoring crucial for managing the syndrome. 

Note: There is no specific mention of the "E" or "N" genes in relation to Cowden syndrome. Therefore, I will only discuss the P

In [ ]:
import re

# =========================
# GOLD STANDARD GENE SET
# =========================

GOLD_GENES = {

    # 🧬 Breast / Ovarian Cancer (Hereditary)
    "hereditary breast cancer": ["BRCA1", "BRCA2"],
    "hereditary ovarian cancer": ["BRCA1", "BRCA2"],
    "hboc": ["BRCA1", "BRCA2"],
    "triple-negative breast cancer": ["BRCA1"],
    "brca-negative hereditary breast cancer": ["PALB2", "CHEK2", "ATM"],
    "early onset breast cancer": ["BRCA1", "BRCA2", "TP53"],

    # 🧬 Colorectal / GI Cancer
    "lynch syndrome": ["MLH1", "MSH2", "MSH6", "PMS2", "EPCAM"],
    "familial adenomatous polyposis": ["APC"],
    "hereditary colorectal cancer": ["MLH1", "MSH2", "MSH6", "PMS2", "EPCAM", "APC"],
    "non-polyposis colorectal cancer": ["MLH1", "MSH2", "MSH6", "PMS2", "EPCAM"],
    "mutyh-associated polyposis": ["MUTYH"],

    # 🧬 Endocrine / Thyroid / Multiple Tumors
    "multiple endocrine neoplasia type 1": ["MEN1"],
    "men1": ["MEN1"],
    "men2": ["RET"],
    "medullary thyroid carcinoma": ["RET"],
    "cowden syndrome": ["PTEN"],
    "endocrine tumors and pheochromocytoma": ["RET", "VHL", "SDHB", "SDHC", "SDHD"],
    "familial medullary thyroid carcinoma": ["RET"],

    # 🧬 Pediatric / Rare Tumor Syndromes
    "li-fraumeni syndrome": ["TP53"],
    "hereditary retinoblastoma": ["RB1"],
    "neurofibromatosis type 1": ["NF1"],
    "neurofibromatosis type 2": ["NF2"],
    "wilms tumor": ["WT1"],

    # 🧬 Kidney / Skin Cancer
    "hereditary renal cell carcinoma": ["VHL", "MET", "FH", "FLCN"],
    "von hippel–lindau disease": ["VHL"],
    "von hippel-lindau disease": ["VHL"],
    "hereditary melanoma": ["CDKN2A"],
    "birt-hogg-dubé syndrome": ["FLCN"],

    # 🧬 Cardiac Genetics
    "hypertrophic cardiomyopathy": ["MYH7", "MYBPC3"],
    "long qt syndrome": ["KCNQ1", "KCNH2", "SCN5A"],
    "arrhythmogenic cardiomyopathy": ["PKP2", "DSP", "DSG2", "DSC2"],
    "familial dilated cardiomyopathy": ["TTN", "LMNA"],

    # 🧬 Neuro / Developmental Disorders
    "spinal muscular atrophy": ["SMN1"],
    "duchenne muscular dystrophy": ["DMD"],
    "rett syndrome": ["MECP2"],
    "fragile x syndrome": ["FMR1"],

    # 🧬 Hemoglobin / Blood Disorders
    "sickle cell disease": ["HBB"],
    "beta-thalassemia": ["HBB"],
    "hereditary hemochromatosis": ["HFE"],
    "hemophilia a": ["F8"],

    # 🧬 Metabolic / Inborn Errors
    "phenylketonuria": ["PAH"],
    "familial hypercholesterolemia": ["LDLR", "APOB", "PCSK9"],
    "wilson disease": ["ATP7B"],
    "glycogen storage disease type i": ["G6PC"],

    # 🔥 HARD / TRICK QUESTIONS
    "hereditary cancer predisposition": [
        "BRCA1", "BRCA2", "TP53", "PTEN", "APC",
        "MLH1", "MSH2", "MSH6", "PMS2", "EPCAM"
    ],
    "breast cancer and gastric cancer": ["CDH1"],
    "sarcomas and brain tumors": ["TP53"],
    "dna mismatch repair": ["MLH1", "MSH2", "MSH6", "PMS2", "EPCAM"],
    "autosomal dominant cancer predisposition in children": ["TP53"]
}

# =========================
# UTILITY FUNCTIONS
# =========================

def normalize(text):
    return text.lower()

def score_answer(predicted, gold):
    predicted = set(predicted)
    gold = set(gold)

    if predicted == gold:
        return 5
    if predicted & gold and len(predicted - gold) == 0:
        return 4
    if predicted & gold:
        return 2
    return 0



In [ ]:
def compare_models(user_question, explain):
    question_key = normalize(user_question)

    # Find gold genes
    gold = []
    for k in GOLD_GENES:
        if k in question_key:
            gold = GOLD_GENES[k]
            break

    # Generate outputs
    base_out = generate(base_model, user_question, explain=False)
    ft_out   = generate(ft_model, user_question, explain=False)

    # ✅ FIX HERE
    base_genes = extract_genes_only(base_out)
    ft_genes   = extract_genes_only(ft_out)

    base_score = score_answer(base_genes, gold)
    ft_score   = score_answer(ft_genes, gold)

    verdict = "🟢 Fine-Tuned model better" if ft_score > base_score else "🔴 Base model better"

    explanation = ""
    if explain and ft_genes:
        explanation = explain_with_base_model(ft_genes, user_question)

    return (
        "\n".join(base_genes),
        "\n".join(ft_genes),
        base_score,
        ft_score,
        verdict,
        explanation
    )


In [ ]:
with gr.Blocks(title="Clinical Genomics AI – Base vs Fine-Tuned") as demo:
    gr.Markdown("##  Clinical Genomics AI – Base vs Fine-Tuned Model")
    gr.Markdown(
        "Gene-level comparison with automatic scoring.\n\n"
        "🟢 Fine-tuned model is optimized for clinical precision."
    )

    # -------------------------
    # Question Input
    # -------------------------
    question = gr.Textbox(
        label="Clinical Genetics Question",
        placeholder="e.g. What genes are associated with Lynch syndrome?",
        lines=2
    )

    explain = gr.Checkbox(
        label="Explanation mode (educational)",
        value=False
    )

    # -------------------------
    # Model Outputs (BIGGER)
    # -------------------------
    with gr.Row():
        base_out = gr.Textbox(
            label="🔴 Base Model Output",
            lines=14
        )
        ft_out = gr.Textbox(
            label="🟢 Fine-Tuned Model Output",
            lines=14
        )

    # -------------------------
    # Explanation (BELOW outputs)
    # -------------------------
    explanation_box = gr.Textbox(
        label="📘 Gene Explanation",
        lines=8
    )

    # -------------------------
    # Scores
    # -------------------------
    with gr.Row():
        base_score = gr.Textbox(
            label="⭐ Base Model Score",
            lines=1
        )
        ft_score = gr.Textbox(
            label="⭐ Fine-Tuned Model Score",
            lines=1
        )

    # -------------------------
    # Verdict
    # -------------------------
    verdict = gr.Textbox(
        label="🏆 Verdict",
        lines=1
    )

    # -------------------------
    # Button at the BOTTOM
    # -------------------------
    run_btn = gr.Button("Compare Models")

    run_btn.click(
        fn=compare_models,
        inputs=[question, explain],
        outputs=[
            base_out,
            ft_out,
            base_score,
            ft_score,
            verdict,
            explanation_box
        ]
    )

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://30eea1e390e6950cf7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://30eea1e390e6950cf7.gradio.live
